# Sprint 2 Live — Core-concept workshops
Use this instructor Colab for LS05–LS08, in a separate practice workspace. Campus Sprints 1–3 use VS Code and two terminals; Sprint 1 uses its Campus Colab only for checkpoint verification. No Campus ZIP or business assignment is needed here.

The public diagnosis example is a small vehicle for the concept. Inspect the same `service/fieldcare/` source layout used locally. The supplied `live_*.py` modules make each registration/attachment visible and run in separate owned processes; in Campus those lines go in `main.py`. Each start stops only our previous process. Never restart during a quota-renewal experiment.

Run setup once. It clones `main`, installs the lock, preserves an existing marked Live folder, and performs no provider call. `ServiceProcess` starts a loopback Uvicorn child with a bounded readiness check. A Colab `127.0.0.1` address belongs to Colab, not your laptop. No foreground Uvicorn call or public tunnel is needed.

See [local setup](https://github.com/richhiey/ai-app-dev_Mod-B/blob/main/docs/local-development.md) and [Live handoff](https://github.com/richhiey/ai-app-dev_Mod-B/blob/main/docs/live-handoff.md).

**Continue in a fresh runtime:** download the Live ZIP and save the notebook after each session. Upload your own ZIP through the Colab Files panel (or use its local path in Jupyter), set `LIVE_ARCHIVE` below, and fill both `RESTORED_MODELS` from your saved model-choice record. Choose a `LIVE_FOLDER` that does not exist. Setup validates the ZIP and restores your edited source, router registrations and selected evidence without running it. Review that source before starting a service. After a successful restore, clear `LIVE_ARCHIVE` before rerunning setup in the same runtime so it reuses the marked folder. Credentials and indexes are excluded: re-enter provider credentials and prepare the index only when the session requires them. A Campus checkpoint cannot be restored here.


## 1. Set up once

In [ ]:
from pathlib import Path
import os, sys, subprocess, tempfile, json, secrets, time
import httpx
REPOSITORY = "https://github.com/richhiey/ai-app-dev_Mod-B.git"
REPO = Path(tempfile.mkdtemp(prefix="module-b-live-source-")) / "repo"
subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", REPOSITORY, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO), "-c", str(REPO / "requirements.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))
from module_b.live import prepare_live_workspace, export_live_workspace, restore_live_workspace
from module_b.runtime import ServiceProcess
# On a fresh runtime, upload your own Live ZIP using the Files panel.
LIVE_ARCHIVE = ""  # Example: "/content/sprint-2-live-123.zip"; keep empty for new practice.
LIVE_FOLDER = Path.cwd() / "fieldcare-sprint-2-live"
# When restoring, explicitly select the same approved models used in that work.
RESTORED_MODELS = {"OPENROUTER_MODEL": "", "OPENROUTER_MODEL_V2": ""}
from module_b.openrouter import CHAT_MODELS, DEFAULT_CHAT_MODEL
if LIVE_ARCHIVE:
    if any(model not in CHAT_MODELS for model in RESTORED_MODELS.values()):
        raise ValueError("Fill both RESTORED_MODELS with your saved approved model choices before restoring.")
    PROJECT = restore_live_workspace(LIVE_ARCHIVE, LIVE_FOLDER)
else:
    PROJECT = prepare_live_workspace(REPO, LIVE_FOLDER)
CALLERS = {"dispatch": "FIELDCARE_DISPATCH_KEY", "partner": "FIELDCARE_PARTNER_KEY"}
caller_keys = {name: secrets.token_urlsafe(32) for name in CALLERS}
previous_models = {name: globals().get("service_env", {}).get(name) for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}
service_env = {variable: caller_keys[name] for name, variable in CALLERS.items()}
service_env["FIELDCARE_WORK_DIR"] = str((PROJECT / "var").resolve())
for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2"):
    service_env[name] = (RESTORED_MODELS[name] if LIVE_ARCHIVE else
                         previous_models[name] or os.environ.get(name, DEFAULT_CHAT_MODEL))
    if service_env[name] not in CHAT_MODELS:
        raise ValueError(f"Select an approved {name}.")
print("Configured models:", {name: service_env[name] for name in RESTORED_MODELS})
if globals().get("service") is not None:
    service.stop()
service = None
supported = {"question": "Which filter and airflow checks are documented?", "equipment_id": "EQ-FC-1002"}
incomplete = {"question": "Which filter checks are documented?"}
print("Live workspace:", PROJECT)
print("Source main:", subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())
print("No provider or index is needed for health, validation and clarification.")

from module_b.live import start_live_service

In [ ]:
print("Caller labels:", list(CALLERS))
print("Temporary values remain in memory; values are never displayed.")


## LS05 — Recognize the caller before application work
Read the route inventory and supplied guard in `live_auth.py`. Registration precedes the static POST inventory. `POLICY = None` means authentication only. Temporary caller values are in memory; the service provider key has a different destination. In local Campus, configuration names live in source and values live in `.env`.

In [ ]:
print((PROJECT / "service/fieldcare/security_settings.py").read_text())
print((PROJECT / "service/fieldcare/live_auth.py").read_text())

In [ ]:
service = start_live_service(PROJECT, "live_auth", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    paths = [path for path, methods in client.get("/openapi.json").json()["paths"].items() if "post" in methods]
    for path in paths:
        for label, key, body in [
            ("missing", None, incomplete), ("wrong", "synthetic-invalid", incomplete),
            ("admitted invalid", caller_keys["dispatch"], {}),
            ("admitted clarification", caller_keys["dispatch"], incomplete),
            ("second caller", caller_keys["partner"], incomplete),
        ]:
            response = client.post(path, json=body, headers={"X-API-Key": key} if key else {})
            print(path, label, response.status_code, response.json().get("status"))
    for path in ["/health", "/docs", "/openapi.json"]:
        print("public GET", path, client.get(path).status_code)

In [ ]:
service.stop()
missing_env = {**service_env, "FIELDCARE_PARTNER_KEY": None}
try:
    rejected_start = start_live_service(PROJECT, "live_auth", missing_env)
except RuntimeError:
    print("Startup refused with a required caller value absent.")
else:
    rejected_start.stop()
    raise AssertionError("Expected configuration refusal")
service = start_live_service(PROJECT, "live_auth", service_env)
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    response = client.post("/v1/diagnose", json=incomplete, headers={"X-API-Key": caller_keys["dispatch"]})
    print("Restored configuration", response.status_code, response.json()["status"])

### Optional provider preparation — after the deterministic checks
The same explicit preparation as Campus: stop the service, load the private provider key, then run `python -m fieldcare.prepare_index`. Embedding the supplied synthetic documents uses provider quota. Set the switch only when you intend to do this. A healthy process alone proves neither provider access nor answer quality. In local Campus, the key belongs in private `.env`; in this instructor Colab, use Colab Secrets or the hidden prompt. Restart the relevant service cell after preparation.

In [ ]:
PREPARE_PROVIDER = False
if PREPARE_PROVIDER:
    if service is not None:
        service.stop()
    from module_b.openrouter import require_openrouter_key
    require_openrouter_key(prompt=True)
    preparation_env = {**os.environ, **service_env,
                       "PYTHONPATH": str(PROJECT / "service") + os.pathsep + str(REPO / "src")}
    subprocess.run([sys.executable, "-m", "fieldcare.prepare_index"],
                   cwd=PROJECT, env=preparation_env, check=True)
else:
    print("Provider preparation pending; deterministic checks remain available.")

In [ ]:
RUN_PROVIDER = False
if RUN_PROVIDER:
    service = start_live_service(PROJECT, "live_auth", service_env, service)
    with httpx.Client(base_url=service.base_url, timeout=120) as client:
        response = client.post("/v1/diagnose", json=supported, headers={"X-API-Key": caller_keys["dispatch"]})
        print(response.status_code, response.json())
else:
    print("Provider comparison pending; the access matrix uses deterministic clarification.")

## LS06 — Count admitted attempts and observe renewal
Guided policy: two admissions per caller per 60 seconds, pooled across protected routes. Campus independent practice later uses three; its assessment uses four. Inspect `live_limits.py` before starting it. The experiment uses clarification and malformed bodies so it buys no model calls. A 422 consumes an admission; 401 and 429 do not add another. Keep one process running through exhaustion, isolation and real-time renewal.

In [ ]:
print((PROJECT / "service/fieldcare/live_limits.py").read_text())

In [ ]:
service = start_live_service(PROJECT, "live_limits", service_env, service)
print("Owned HTTP service:", service.base_url)

In [ ]:
sequence_started = time.monotonic()
with httpx.Client(base_url=service.base_url, timeout=15) as client:
    sequence = [
        ("missing", None, "/v1/diagnose", incomplete),
        ("A invalid counts", "dispatch", "/v1/diagnose", {}),
        ("A second admission", "dispatch", "/v2/diagnose", incomplete),
        ("A exhausted", "dispatch", "/v1/diagnose", incomplete),
        ("B independent", "partner", "/v1/diagnose", incomplete),
    ]
    for label, caller, path, body in sequence:
        headers = {"X-API-Key": caller_keys[caller]} if caller else {}
        response = client.post(path, json=body, headers=headers)
        print(round(time.monotonic() - sequence_started, 3), label, response.status_code,
              "Retry-After:", response.headers.get("retry-after"))
        if response.status_code == 429:
            retry_after = int(response.headers["retry-after"])
            retry_at = time.monotonic() + retry_after
print("Keep this process running. Return to the next cell after Retry-After seconds.")

In [ ]:
remaining = max(0, retry_at - time.monotonic())
if remaining > 0:
    print("Wait", round(remaining, 1), "more seconds, then rerun this cell; do not restart.")
else:
    with httpx.Client(base_url=service.base_url, timeout=15) as client:
        response = client.post("/v1/diagnose", json=incomplete, headers={"X-API-Key": caller_keys["dispatch"]})
        print("Same-process renewal", response.status_code, response.json())

### Independent changed-order experiment
Write a short sequence and predict every result before editing a copy of `code-09` in this same notebook. Start caller B first, include an admitted invalid body, use two paths and demonstrate renewal. Restart `code-08` once to begin a new experiment, then keep that worker alive. Record timestamps and statuses. A restart empties all buckets and cannot prove renewal. Leave Campus's independent sequence for your own local practice.

## LS07 — Follow a synthetic marker through three local sinks

This exercise makes no provider call and sends no HTTP request. Use only the visibly synthetic marker. A disposable local folder holds an intentionally exposed log policy, prompt policy, and Git history. Run the before check, apply each prevention, then repeat the workflow with a fresh `RUN_NAME`.


In [ ]:
from module_b.secret_workshop import prepare_secret_workshop

RUN_NAME = "attempt_1"  # Change this to create a separate fresh fixture.
MARKER = f"SYNTHETIC_{RUN_NAME.upper()}_NOT_A_CREDENTIAL"
FIXTURE = prepare_secret_workshop(
    PROJECT / "secret-fixtures" / RUN_NAME,
    marker=MARKER,
    repo_root=REPO,
)
checks = {}
print("Disposable local fixture:", FIXTURE)
print("Provider calls:", 0, "| HTTP requests:", 0)


### Before prevention — inspect the actual local artifacts

The starter `log_policy.py` and `prompt_policy.py` deliberately pass too much data. Run their real functions with synthetic values. The local exception below is only a Python input to the logger; it is not an HTTP or provider failure.


In [ ]:
import json
import runpy

event = {
    "request_id": "synthetic-local-attempt-01",
    "route": "/v1/diagnose",
    "outcome": "completed",
    "headers": {"X-API-Key": MARKER},
}
log_record = runpy.run_path(str(FIXTURE / "log_policy.py"))["log_record"]
ordinary = log_record(event)
failed = log_record(
    {**event, "request_id": "synthetic-local-attempt-02", "outcome": "failed"},
    ValueError("Synthetic local exception contains " + MARKER),
)
LOG_PATH = FIXTURE / "observed.log"
LOG_PATH.write_text(json.dumps([ordinary, failed], indent=2) + "\n")

build_prompt = runpy.run_path(str(FIXTURE / "prompt_policy.py"))["build_prompt"]
payload = build_prompt(
    "Which filter check should the technician make?",
    "Synthetic HX guidance: inspect the intake filter.",
    {"provider_key": MARKER},
)
PAYLOAD_PATH = FIXTURE / "outbound-payload.json"
PAYLOAD_PATH.write_text(json.dumps(payload, indent=2) + "\n")

log_text = LOG_PATH.read_text()
payload_text = PAYLOAD_PATH.read_text()
print("Marker in saved ordinary/error logs:", MARKER in log_text)
print("Marker in locally built prompt payload:", MARKER in payload_text)
print("Saved ordinary/error log:")
display(json.loads(log_text))
print("Locally assembled outbound payload:")
display(json.loads(payload_text))


### Prevent disclosure in logs

Open `examples/patterns/secret_workshop/log_safe.py` and adapt its field selection in `FIXTURE/log_policy.py` using Colab’s file editor. Keep the request ID, route, outcome, and a bounded error code; drop headers and exception text. Then rerun the next cell and inspect `observed.log`.


In [ ]:
log_record = runpy.run_path(str(FIXTURE / "log_policy.py"))["log_record"]
ordinary = log_record(event)
failed = log_record(
    {**event, "request_id": "synthetic-local-attempt-02", "outcome": "failed"},
    ValueError("Synthetic local exception contains " + MARKER),
)
LOG_PATH.write_text(json.dumps([ordinary, failed], indent=2) + "\n")
log_text = LOG_PATH.read_text()
checks["ordinary and error logs exclude the marker"] = MARKER not in log_text
checks["safe diagnostic fields remain"] = all(
    row.get("request_id") and row.get("route") == "/v1/diagnose"
    and row.get("outcome") in {"completed", "failed"}
    for row in json.loads(log_text)
)
print("Marker in saved logs:", MARKER in log_text)
print("Retained record fields:", sorted(ordinary))


### Keep runtime configuration out of the prompt

Adapt `examples/patterns/secret_workshop/prompt_safe.py` in `FIXTURE/prompt_policy.py`. The prompt needs the question and task context, not runtime configuration. No request is sent to a model.


In [ ]:
build_prompt = runpy.run_path(str(FIXTURE / "prompt_policy.py"))["build_prompt"]
payload = build_prompt(
    "Which filter check should the technician make?",
    "Synthetic HX guidance: inspect the intake filter.",
    {"provider_key": MARKER},
)
PAYLOAD_PATH.write_text(json.dumps(payload, indent=2) + "\n")
payload_text = PAYLOAD_PATH.read_text()
checks["outbound prompt excludes the marker"] = MARKER not in payload_text
checks["question and useful context remain"] = (
    "Which filter check" in payload_text and "inspect the intake filter" in payload_text
)
print("Marker in outbound payload:", MARKER in payload_text)
print("Question and context retained:", checks["question and useful context remain"])


### Stop future Git exposure; keep history distinct

This disposable repository has a synthetic `.env` in its initial commit and a new `local.key`. Add both patterns to `.gitignore`, then use the visible `git rm --cached` command to remove `.env` from the current index while leaving the local file in place. This does not erase the initial commit.


In [ ]:
(FIXTURE / ".gitignore").write_text(".env\nlocal.key\n")
subprocess.run(
    ["git", "-C", str(FIXTURE), "rm", "--cached", "--ignore-unmatch", ".env"],
    check=True, capture_output=True, text=True,
)
tracked = subprocess.run(
    ["git", "-C", str(FIXTURE), "ls-files", "--error-unmatch", ".env"],
    capture_output=True, text=True,
).returncode == 0
ignored_env = subprocess.run(
    ["git", "-C", str(FIXTURE), "check-ignore", "-q", ".env"]
).returncode == 0
ignored_key = subprocess.run(
    ["git", "-C", str(FIXTURE), "check-ignore", "-q", "local.key"]
).returncode == 0
initial_env = subprocess.run(
    ["git", "-C", str(FIXTURE), "show", "HEAD:.env"],
    check=True, capture_output=True, text=True,
).stdout
checks.update({
    "new .env is ignored": ignored_env,
    "new local.key is ignored": ignored_key,
    ".env is no longer tracked": not tracked,
    "initial commit still contains the marker": MARKER in initial_env,
})
print(".env ignored / tracked:", ignored_env, tracked)
print("local.key ignored:", ignored_key)
print("Marker remains in initial commit:", MARKER in initial_env)


### Record a fresh-marker check

Change `RUN_NAME` in the fixture cell to a new value and repeat the before check, both code edits, and the Git prevention. Then run the summary below. It checks the artifacts created in that fresh workspace; it does not claim to scan your actual project.

This summary saves your two edited policy files and the selected synthetic checks/results under `evidence/`. Rerun it after your final edits, then run the notebook’s export cell and download the ZIP. The disposable `.git`, `.env`, `local.key` and raw fixture folder are excluded. Save notebook notes as well.


In [ ]:
for name, passed in checks.items():
    print(("PASS" if passed else "REVIEW"), name)
print("Evidence kind: local synthetic fixture")
print("Provider calls:", 0, "| HTTP requests:", 0)

# Preserve only the two edited policies and selected synthetic results in the Live ZIP.
import shutil
POLICY_EVIDENCE = PROJECT / "evidence/ls07-policies" / RUN_NAME
POLICY_EVIDENCE.mkdir(parents=True, exist_ok=True)
for name in ("log_policy.py", "prompt_policy.py"):
    shutil.copy2(FIXTURE / name, POLICY_EVIDENCE / name)
(PROJECT / "evidence" / f"ls07-{RUN_NAME}.json").write_text(json.dumps({
    "evidence_kind": "local synthetic fixture", "run_name": RUN_NAME,
    "checks": checks, "logs": json.loads(LOG_PATH.read_text()),
    "outbound_payload": json.loads(PAYLOAD_PATH.read_text()),
    "provider_calls": 0, "http_requests": 0,
}, indent=2) + "\n")
print("Retained for export:", POLICY_EVIDENCE)


## LS08 — Review a control claim
Use your LS05 request matrix, the continuous LS06 timeline and LS07 synthetic evidence. Demonstrate one claim and its source location. Reviewers challenge a missing observation, not the private Campus assignment. Repair one narrow issue or add a check. Label provider work separately; access rejection follows from the guard's source and HTTP result, not a fabricated adapter counter.
After a fresh runtime restore, open `evidence/ls07-<RUN_NAME>.json` and `evidence/ls07-policies/<RUN_NAME>/` for the saved check and edited policies. Their observations are synthetic; a fresh claim requires rerunning a new fixture.


In [ ]:
if service is not None:
    service.stop()
(PROJECT / "evidence").mkdir(exist_ok=True)
(PROJECT / "evidence/live-models.json").write_text(json.dumps(
    {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")}, indent=2) + "\n")
print("Save these configured model choices with your notebook:",
      {name: service_env[name] for name in ("OPENROUTER_MODEL", "OPENROUTER_MODEL_V2")})
CHECKPOINT_OUT = PROJECT.parent / f"sprint-2-live-{time.time_ns()}.zip"
print("Download:", export_live_workspace(PROJECT, CHECKPOINT_OUT))

### Carry forward

Keep this Live workspace separate from Campus. Continue your local Campus project in place; Sprint 3 Live starts a separate public practice workspace.